In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [2]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
210,NaN,2025-26,1629599,Amir Coffey,Amir,1610612756,PHX,Phoenix Suns,22501185,2026-04-10T00:00:00,PHX @ LAL,L,20.550000,3,6,0.500,1,2,0.500,0,0,0.000,0,2,2,0,1,1,0,0,2,0,7,-11,11.4,0,0,12.0,1,20:33,1,82.4,82.9,82.9,114.2,115.4,115.4,-31.8,-32.5,-32.5,0.000,0.0,0.0,0.000,0.118,0.051,14.3,14.3,0.583,0.583,0.146,0.155,94.22,93.43,77.86,93.43,0.056,41,3.0,6.0,26,77,0.338,7,40,0.175,14,21,0.667,18,29,47,17,23.0,8,1,3,18,18,73,-28.0,80.0,81.1,113.2,112.2,-33.2,-31.1,0.654,0.74,13.2,0.400,0.816,0.570,0.256,0.383,0.423,90.2,90.0,75.00,90,0.255,1610612747,LAL,Los Angeles Lakers,35,69,0.507,8,20,0.400,23,30,0.767,4,31,35,27,11.0,17,3,1,18,18,101,28.0,113.2,112.2,80.0,81.1,33.2,31.1,0.771,2.45,21.8,0.184,0.600,0.430,0.122,0.565,0.614,90.2,90.0,75.00,90,0.745,NaN,SG,28.0
211,NaN,2025-26,1627739,Kris Dunn,Kris,1610612746,LAC,LA Clippers,22501183,2026-04-10T00:00:00,LAC @ POR,L,26.566667,2,3,0.667,1,2,0.500,0,0,0.000,1,4,5,1,2,0,0,0,4,1,5,3,10.5,0,0,12.0,1,26:34,1,128.6,131.4,131.4,117.5,123.1,123.1,11.2,8.3,8.3,0.043,0.5,16.7,0.042,0.143,0.096,33.3,33.3,0.833,0.833,0.088,0.086,96.26,93.05,77.54,93.05,0.031,51,2.0,3.0,36,83,0.434,13,40,0.325,12,12,1.000,10,25,35,17,17.0,6,3,7,27,15,97,-19.0,101.8,102.1,116.7,122.1,-14.9,-20.0,0.472,1.00,13.8,0.250,0.574,0.411,0.179,0.512,0.549,97.3,95.0,79.17,95,0.350,1610612757,POR,Portland Trail Blazers,37,83,0.446,12,39,0.308,30,35,0.857,14,32,46,23,15.0,12,7,3,15,27,116,19.0,116.7,122.1,101.8,102.1,14.9,20.0,0.622,1.53,16.8,0.426,0.750,0.589,0.158,0.518,0.589,97.3,95.0,79.17,95,0.650,G,PG,31.0
212,NaN,2025-26,1641771,Jalen Slawson,Jalen,1610612754,IND,Indiana Pacers,22501175,2026-04-10T00:00:00,IND vs. PHI,L,15.183333,0,4,0.000,0,2,0.000,2,2,1.000,2,3,5,3,5,0,1,0,2,3,2,-14,10.5,0,0,12.0,1,15:11,1,59.0,59.4,59.4,101.9,100.0,100.0,-42.8,-40.6,-40.6,0.500,0.6,23.1,0.111,0.125,0.119,38.5,38.8,0.000,0.205,0.270,0.273,102.11,102.74,85.62,102.74,-0.041,32,0.0,4.0,33,88,0.375,14,50,0.280,14,16,0.875,10,42,52,25,21.0,6,8,2,15,14,94,-11.0,88.6,89.5,100.6,101.0,-12.0,-11.4,0.758,1.19,17.5,0.228,0.723,0.492,0.200,0.455,0.495,105.2,104.5,87.08,105,0.456,1610612755,PHI,Philadelphia 76ers,42,104,0.404,5,29,0.172,16,19,0.842,16,42,58,17,8.0,13,2,8,14,15,105,11.0,100.6,101.0,88.6,89.5,12.0,11.4,0.405,2.13,12.2,0.277,0.772,0.508,0.077,0.428,0.467,105.2,104.5,87.08,104,0.544,NaN,SF,26.0
205,NaN,2025-26,1642362,Payton Sandfort,Payton,161

### Load latest odds on file

In [3]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260411_140648.json


,home_team,away_team,commence_time,bookmakers
0,Miami Heat,Atlanta Hawks,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Boston Celtics,Orlando Magic,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Toronto Raptors,Brooklyn Nets,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,New York Knicks,Charlotte Hornets,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Cleveland Cavaliers,Washington Wizards,2026-04-12 22:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [4]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-11 14:06:48
US latest pull: 2026-04-11 14:05:22


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,Bam Adebayo,Over,20.5,-137,2026-04-12,2026-04-11T21:04:42Z,2026-04-11 14:06:48
1,Underdog,player_points,Bam Adebayo,Under,20.5,-137,2026-04-12,2026-04-11T21:04:42Z,2026-04-11 14:06:48
2,Underdog,player_points,Tyler Herro,Over,20.5,-137,2026-04-12,2026-04-11T21:04:42Z,2026-04-11 14:06:48
3,Underdog,player_points,Tyler Herro,Under,20.5,-137,2026-04-12,2026-04-11T21:04:42Z,2026-04-11 14:06:48
4,Underdog,player_points,Jaime Jaquez Jr,Over,13.5,-137,2026-04-12,2026-04-11T21:04:42Z,2026-04-11 14:06:48


### Load my models

In [5]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]
min_scaler = min_bundle.get("scaler")

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]
ppm_scaler = ppm_bundle.get("scaler")
#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]
apm_scaler = apm_bundle.get("scaler")

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]
rpm_scaler = rpm_bundle.get("scaler")

### Get Min predictions and Stat Per Min predictions 

In [6]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Bam Adebayo,AST,12.92,32.20,40.21,0.0461,0.1343,0.2388,0.60,4.32,9.60,"[0.071599045346062, 0.057175528873642, 0.02598..."
1,Davion Mitchell,AST,18.00,29.12,37.76,0.0842,0.2067,0.3159,1.52,6.02,11.93,"[0.2074688796680497, 0.18796992481203, 0.15686..."
2,Darius Garland,AST,14.52,30.15,39.80,0.0975,0.2227,0.3459,1.41,6.71,13.77,"[0.2307185234014502, 0.2556455048998722, 0.232..."
3,Deni Avdija,AST,15.45,32.27,40.29,0.0758,0.1787,0.2929,1.17,5.77,11.80,"[0.2486016159105034, 0.2355404344412457, 0.148..."
4,Jaime Jaquez Jr.,AST,16.50,26.83,35.79,0.0370,0.1611,0.2837,0.61,4.32,10.15,"[0.2383384405856315, 0.2869440459110473, 0.220..."
5,Kawhi Leonard,AST,11.98,30.10,39.65,0.0345,0.1255,0.2094,0.41,3.78,8.30,"[0.1586294416243654, 0.0908265213442325, 0.0, ..."
6,LeBron James,AST,12.65,31.94,41.05,0.1341,0.2546,0.3958,1.70,8.13,16.25,"[0.2106530243755642, 0.1243162605668821, 0.149..."
7,Jrue Holiday,AST,13.78,30.06,39.31,0.0688,0.1905,0.3020,0.95,5.73,11.87,"[0.2501563477173233, 0.0823723228995057, 0.183..."
8,Precious Achiuwa,AST,14.21,28.04,36.64,-0.0001,0.0644,0.1419,-0.00,1.81,5.20,"[0.0, 0.0677966101694915, 0.0273373428102788, ..."
9,Nickeil Alexander-Walker,AST,17.16,31.52,40.84,0.0330,0.1278,0.2107,0.57,4.03,8.61,"[0.0293944738389182, 0.1357588922074395, 0.065..."


### Get Line Probabilities

In [7]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
37,Deni Avdija,PTS,26.5,15.45,32.27,40.29,5.79,21.30,42.47,0.353,0.647
49,Dyson Daniels,PTS,12.5,15.63,29.04,38.44,2.84,11.10,27.14,0.438,0.562
52,Deandre Ayton,PTS,12.5,10.50,26.66,36.49,2.73,13.21,32.29,0.491,0.509
48,Jonathan Kuminga,PTS,13.5,15.78,21.48,30.85,2.63,10.91,28.30,0.464,0.536
45,CJ McCollum,PTS,18.5,13.62,28.16,37.63,4.93,17.81,40.74,0.498,0.502
14,Toumani Camara,REB,4.5,15.93,30.85,40.47,1.05,4.92,11.71,0.414,0.586
23,Deandre Ayton,REB,7.0,10.50,26.66,36.49,1.77,7.99,16.08,0.356,0.563
46,Onyeka Okongwu,PTS,15.5,15.72,26.85,36.04,3.74,11.52,26.37,0.273,0.727
1,Davion Mitchell,AST,6.5,18.00,29.12,37.76,1.52,6.02,11.93,0.448,0.552
0,Bam Adebayo,AST,3.5,12.92,32.20,40.21,0.60,4.32,9.60,0.573,0.427


In [8]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
0,Bam Adebayo,AST,3.5,12.92,32.20,40.21,0.60,4.32,9.60,0.573,0.427,AST,Underdog,Atlanta Hawks,4.8,242.5,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,5.1,5.0,2.33,1.6,1.5,-0.687,0.754,0.246,30.44,-57.44,0.6,0.6,0.47,0.51,34.19,5.31,0.23,0.06,2.67,6.0
40,Maxime Raynaud,PTS,15.5,21.65,32.28,37.86,6.20,17.45,35.76,0.616,0.384,PTS,Underdog,Portland Trail Blazers,16.5,227.5,113.6,11.0,101.67,9.0,-137.0,-137.0,0.578,0.578,16.4,16.5,6.69,0.9,1.0,-0.135,0.554,0.446,-4.16,-22.85,0.6,0.6,0.60,0.32,31.72,5.20,0.20,0.06,17.33,3.0
10,Andrew Wiggins,REB,4.5,14.11,26.88,37.67,0.56,3.62,9.31,0.348,0.651,REB,Underdog,Atlanta Hawks,4.8,242.5,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,3.6,3.0,2.22,-0.9,-1.5,0.405,0.343,0.657,-40.66,13.66,0.2,0.3,0.40,0.49,27.19,6.71,0.17,0.04,5.50,6.0
14,Toumani Camara,REB,4.5,15.93,30.85,40.47,1.05,4.92,11.71,0.414,0.586,REB,Underdog,Sacramento Kings,-16.5,227.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,4.4,4.5,1.84,-0.1,0.0,0.054,0.478,0.522,-17.31,-9.70,0.4,0.5,0.40,0.60,32.93,5.32,0.18,0.05,5.86,7.0
35,Scoot Henderson,PTS,15.5,15.47,29.39,37.95,3.96,15.54,31.54,0.508,0.491,PTS,Underdog,Sacramento Kings,-16.5,227.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,15.5,14.5,4.86,0.0,-1.0,0.000,0.500,0.500,-13.50,-13.50,0.4,0.4,0.40,0.32,27.60,5.81,0.23,0.05,9.67,3.0


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
48,Jonathan Kuminga,PTS,13.5,15.78,21.48,30.85,2.63,10.91,28.30,0.464,0.536,PTS,PrizePicks,Miami Heat,-4.8,242.5,113.7,13.0,104.22,1.0,-137.0,-137.0,0.578,0.578,10.7,11.0,6.68,-2.8,-2.5,0.419,0.338,0.662,-41.53,14.52,0.4,0.3,0.40,0.48,21.81,3.59,0.20,0.05,15.00,1.0
22,LeBron James,REB,7.5,12.65,31.94,41.05,1.32,6.78,13.30,0.396,0.604,REB,PrizePicks,Utah Jazz,-16.5,237.0,120.8,29.0,103.52,2.0,-137.0,-137.0,0.578,0.578,7.4,7.0,1.78,-0.1,-0.5,0.056,0.478,0.522,-17.31,-9.70,0.4,0.5,0.40,0.37,33.73,3.96,0.24,0.06,5.14,7.0
6,LeBron James,AST,9.5,12.65,31.94,41.05,1.70,8.13,16.25,0.386,0.614,AST,PrizePicks,Utah Jazz,-16.5,237.0,120.8,29.0,103.52,2.0,-137.0,-137.0,0.578,0.578,8.9,9.5,3.98,-0.6,0.0,0.151,0.440,0.560,-23.88,-3.12,0.6,0.5,0.40,0.29,33.73,3.96,0.24,0.06,10.14,7.0
1,Davion Mitchell,AST,6.5,18.00,29.12,37.76,1.52,6.02,11.93,0.448,0.552,AST,PrizePicks,Atlanta Hawks,4.8,242.5,112.6,9.0,102.48,6.0,-137.0,-137.0,0.578,0.578,6.2,6.0,2.62,-0.3,-0.5,0.115,0.454,0.546,-21.46,-5.55,0.4,0.4,0.27,0.36,30.72,6.67,0.16,0.04,6.12,8.0
13,Jrue Holiday,REB,4.5,13.78,30.06,39.31,0.78,4.32,9.45,0.457,0.543,REB,PrizePicks,Sacramento Kings,-16.5,227.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,5.0,5.0,2.49,0.5,0.5,-0.201,0.580,0.420,0.34,-27.34,0.8,0.6,0.60,0.48,31.02,5.27,0.22,0.06,3.00,2.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
19,Nickeil Alexander-Walker,REB,3.5,17.16,31.52,40.84,0.36,2.79,8.00,0.374,0.626,REB,Betr DFS,Miami Heat,-4.8,242.5,113.7,13.0,104.22,1.0,-137.0,-137.0,0.578,0.578,3.4,4.0,1.26,-0.1,0.5,0.079,0.469,0.531,-18.87,-8.14,0.6,0.7,0.60,0.38,35.34,4.38,0.21,0.03,3.40,5.0
42,Devin Carter,PTS,15.5,23.42,30.76,36.83,5.82,16.70,36.20,0.592,0.408,PTS,Betr DFS,Portland Trail Blazers,16.5,227.5,113.6,11.0,101.67,9.0,-137.0,-137.0,0.578,0.578,14.8,15.0,7.87,-0.7,-0.5,0.089,0.465,0.535,-19.56,-7.45,0.4,0.5,0.40,0.12,27.91,6.36,0.23,0.05,0.00,2.0
36,Jrue Holiday,PTS,15.5,13.78,30.06,39.31,3.64,15.69,34.41,0.452,0.548,PTS,Betr DFS,Sacramento Kings,-16.5,227.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,16.5,13.0,7.74,1.0,-2.5,-0.129,0.551,0.449,-4.68,-22.33,0.6,0.4,0.33,0.34,31.02,5.27,0.22,0.06,5.00,2.0
40,Maxime Raynaud,PTS,15.5,21.65,32.28,37.86,6.20,17.45,35.76,0.616,0.384,PTS,Betr DFS,Portland Trail Blazers,16.5,227.5,113.6,11.0,101.67,9.0,-137.0,-137.0,0.578,0.578,16.4,16.5,6.69,0.9,1.0,-0.135,0.554,0.446,-4.16,-22.85,0.6,0.6,0.60,0.32,31.72,5.20,0.20,0.06,17.33,3.0
37,Deni Avdija,PTS,26.5,15.45,32.27,40.29,5.79,21.30,42.47,0.353,0.647,PTS,Betr DFS,Sacramento Kings,-16.5,227.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,24.3,24.5,5.52,-2.2,-2.0,0.399,0.345,0.655,-40.32,13.31,0.6,0.3,0.27,0.25,32.76,6.24,0.30,0.02,21.57,7.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

KeyError: 'CATEGORY'

In [ ]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
39,Toumani Camara,PTS,14.5,15.93,30.85,40.47,3.07,13.16,28.80,0.503,0.497,PTS,Underdog,Sacramento Kings,-16.5,227.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,18.5,17.5,9.54,4.0,3.0,-0.419,0.662,0.338,14.52,-41.53,0.8,0.7,0.60,0.36,32.93,5.32,0.18,0.05,10.29,7.0
17,Jalen Johnson,REB,10.5,13.23,31.12,39.47,1.93,7.56,15.01,0.330,0.670,REB,PrizePicks,Miami Heat,-4.8,242.5,113.7,13.0,104.22,1.0,-137.0,-137.0,0.578,0.578,9.2,10.0,3.12,-1.3,-0.5,0.417,0.338,0.662,-41.53,14.52,0.8,0.5,0.47,0.47,34.41,5.28,0.26,0.04,12.25,4.0
53,Nique Clifford,PTS,16.5,23.98,34.73,40.36,5.20,14.85,31.64,0.373,0.626,PTS,PrizePicks,Portland Trail Blazers,16.5,227.5,113.6,11.0,101.67,9.0,-137.0,-137.0,0.578,0.578,13.4,13.5,6.28,-3.1,-3.0,0.494,0.311,0.689,-46.20,19.19,0.6,0.4,0.33,0.11,33.23,5.76,0.19,0.04,6.67,3.0
38,Donovan Clingan,PTS,12.5,13.49,24.16,34.35,3.11,10.06,25.22,0.428,0.572,PTS,Underdog,Sacramento Kings,-16.5,227.5,120.2,28.0,100.14,17.0,-137.0,-137.0,0.578,0.578,10.6,9.0,5.95,-1.9,-3.5,0.319,0.375,0.625,-35.13,8.12,0.4,0.4,0.53,0.29,26.44,3.52,0.16,0.05,9.67,6.0
42,Devin Carter,PTS,15.5,23.42,30.76,36.83,5.82,16.70,36.20,0.592,0.408,PTS,PrizePicks,Portland Trail Blazers,16.5,227.5,113.6,11.0,101.67,9.0,-137.0,-137.0,0.578,0.578,14.8,15.0,7.87,-0.7,-0.5,0.089,0.465,0.535,-19.56,-7.45,0.4,0.5,0.40,0.12,27.91,6.36,0.23,0.05,0.00,2.0


### Get top EVs for 2 legs

In [14]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 39  |  Pairs: 2  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [15]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 19  |  Pairs: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

NameError: name 'draftKings_all_lines' is not defined

In [17]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 13  |  Pairs: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [18]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 39  |  Triples: 4  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 19  |  Triples: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 13  |  Triples: 0  |  Slate: 0  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

NameError: name 'draftKings_all_lines' is not defined